# EECS 第7回 演習: 簡易インタプリタの実装

スライド: [EECS 第7回 (PDF)](https://3tty0n.github.io/2026-scripting-lab/eecs07-forth/eecs07-forth-beamer.pdf)

**インタプリタ** (プログラムを読んで、その場で実行するプログラム) を作る。
作るのは **Forth** という言語のインタプリタ。Forth は、数と命令を空白で区切って並べるだけの、とても小さな言語である。
Forth の命令1つ1つを**ワード**と呼ぶ。

まず**電卓**を作り、そこに「新しいワードを自分で定義する仕組み」を足して、言語にする。

```
3 4 + .          →  7     (3 と 4 を積み、足して、表示する)
: SQUARE DUP * ;          (SQUARE という新しいワードを定義する)
5 SQUARE .       →  25
```

使う道具は、第6回で使ったものだけ。

- **スタック**: 皿を積み重ねるように、最後に置いたものから取り出すリスト。
  `append` で一番上に積み (push)、`pop` で一番上を取り出す (pop)
- **関数を入れた辞書**: `WORDS['+'] = add` のように、名前から関数を引く表

## このノートブックの使い方

1. **一番上のセルから順に、全部実行する** (`ランタイム → すべてのセルを実行` でもよい)
2. `TODO` と書かれた所の `...` や `pass` が、自分で書く場所。ほとんどは**1行**で書ける
3. 書いたら、そのすぐ下の **確認セル** (先頭が `# [CHECK]`) を実行する。`OK` と出れば正解
4. 確認セルは書き換えない。`NG` や赤いエラーが出たら、表示された行やメッセージを手がかりに直す
5. `NameError` (名前が見つからない) が出たら、上のセルを実行し忘れている。上から実行し直す

## 全体の地図

インタプリタは次の関数でできている。**★が自分で書く部分**で、他は用意してある。

```
forth_eval(stack, text)          用意済み: ワードを左から1つずつ取り出して実行する
 ├─ read_definition(tokens, i)   ★課題3  ': 名前 本体 ;' を読み取る
 ├─ eval_user(stack, word)       ★課題4  自分で定義したワードを実行する
 └─ evaluate(stack, text)        ★課題1  電卓: WORDS を引いて実行する
                                 ★課題2  WORDS のワードを増やす (. DUP DROP SWAP)
repl(lines)                      ★課題5  1行ずつ実行して結果を返すループ
```

| 課題 | 書くもの | 配点 |
|---|---|---|
| 課題1 | 四則演算、`evaluate` | 20 |
| 課題2 | `.` とスタック操作ワード | 15 |
| 課題3 | `read_definition` | 15 |
| 課題4 | `eval_user` | 15 |
| 課題5 | `repl` | 15 |
| 発展1 | 比較演算子 | 10 |
| 発展2 | `IF / ELSE / THEN` | 10 |

---
## 課題0: 用意されたものを読む (採点なし)

次のセルを実行する。**この後の課題で埋める「枠」** がここにある。書き換えなくてよい。

- `WORDS`: 組み込みワードの表 (辞書)。鍵がワードの名前、値がそのワードの関数。課題1・2 で増やす
- `USER_WORDS`: 自分で定義したワードの表。鍵が名前、値が本体 (ワードのリスト)。課題4 で使う
- `forth_eval(stack, text)`: ワードを左から1つずつ実行するループ。
  `:` なら定義を読み、自分で定義したワードなら本体を実行し、それ以外は課題1 の `evaluate` に任せる
- `run(text)`: 空のスタックから実行して、最後のスタックを返す。確認セルはこれを使う
- `trace(text)`: 1語ごとにスタックの様子を表示する。うまく動かないときに使う

```
run('3 4 +')      →  [7]
trace('3 4 +')    →      3  [3]
                         4  [3, 4]
                         +  [7]
```

In [ ]:
WORDS = {}          # 組み込みワード。課題1, 2 で増やす
USER_WORDS = {}     # 自作ワード。名前 -> 本体 (ワードのリスト)

def forth_eval(stack, text):
    """ワードを左から1つずつ実行する。text は文字列でもワードのリストでもよい"""
    tokens = text.split() if isinstance(text, str) else text
    i = 0  # 今見ているワードの位置
    while i < len(tokens):
        token = tokens[i]
        key = token.upper()
        # ':' なら定義を読み取り、USER_WORDS に登録する
        if token == ':':
            name, body, i = read_definition(tokens, i)
            USER_WORDS[name] = body
            continue
        # 発展2 では、ここに IF / ELSE / THEN の処理を足す
        # 自分で定義したワードなら本体を実行。それ以外は evaluate に任せる
        if key in USER_WORDS:
            eval_user(stack, key)
        else:
            evaluate(stack, key)
        i += 1


def run(text):
    """空のスタックから text を実行して、できたスタックを返す"""
    stack = []
    forth_eval(stack, text)
    return stack


def trace(text):
    """1語ごとにスタックを表示する (うまく動かないときの確認用。: ... ; の定義は渡さないこと)"""
    stack = []
    for token in text.split():
        forth_eval(stack, [token])
        print(f'{token:>8}  {stack}')

In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = 'e07'


---
## 課題1: 電卓 (20点)

**やること**: 四則演算のワード `- * /` を作り、`evaluate` の空いている所を埋めて、数式を計算できるようにする。

Forth は**逆ポーランド記法** (演算子を数の後ろに書く書き方) で計算する。
`3 4 +` は「3 を積む、4 を積む、上の2つを足す」。かっこも、計算の順番の決まりも要らない。

| 読んだもの | 行うこと | スタック |
|---|---|---|
| `3` | 3 を積む | `[3]` |
| `4` | 4 を積む | `[3, 4]` |
| `+` | 2つ取り出し、和を積む | `[7]` |
| `2` | 2 を積む | `[7, 2]` |
| `*` | 2つ取り出し、積を積む | `[14]` |

**入出力の例**:

```
run('3 4 +')        →  [7]
run('10 3 -')       →  [7]       (10 - 3)
run('3 4 + 2 *')    →  [14]      ((3 + 4) * 2)
run('20 4 /')       →  [5]
run('3 FOO')        →  ValueError (FOO というワードは無い)
```

**手順** (下のセルの `TODO` と同じ順):

1. `sub`, `mul`, `div` の中身を、`add` と同じ形で書く。割り算は `//` (整数の割り算)
2. `evaluate` の1つ目の `...`: `WORDS` にあるワードなら、その関数にスタックを渡して呼ぶ (`WORDS[token](stack)`)
3. `evaluate` の2つ目の `...`: 数なら、`int(token)` にしてスタックに積む

**ヒント**: `WORDS[token]` は関数そのもの。後ろに `(stack)` を付けると呼び出せる。

**つまずきやすい点**:

- 引き算・割り算の**順序**。先に `pop` されるのは**右側** (後に積んだ方) の値。`b, a = s.pop(), s.pop()` として `a - b`。
  確認セルの `assert run('10 3 -') == [7]` で止まったら、ここを疑う
- `/` を `/` のまま書くと `5.0` (小数) になる。`//` を使う

### 例題

次のセルを実行して、動きを見る。

`3 4 +` を手で追う。スタックは `append` で積み、`pop` で取る。

In [ ]:
# 例題
s = []
s.append(3)
s.append(4)
b = s.pop()     # 後に積んだ 4 が先に出る
a = s.pop()
s.append(a + b)
print(s)    # [7]

s = [10, 3]
b, a = s.pop(), s.pop()
print(a - b)    # 7

`+` の `add` は書いてある。`sub`, `mul`, `div` も同じ形。
`evaluate` は「`WORDS` にあれば呼ぶ・数なら積む・それ以外はエラー」の3通りに分ける。3つ目 (エラー) は書いてある。

In [ ]:
def add(s):
    b, a = s.pop(), s.pop()
    s.append(a + b)


def sub(s):
    b, a = s.pop(), s.pop()
    s.append(a - b)


def mul(s):
    b, a = s.pop(), s.pop()
    s.append(a * b)


def div(s):
    b, a = s.pop(), s.pop()
    s.append(a // b)


WORDS['+'] = add
WORDS['-'] = sub
WORDS['*'] = mul
WORDS['/'] = div


def evaluate(stack, text):
    for token in text.split():
        if token in WORDS:
            WORDS[token](stack)
        elif token.lstrip('-').isdigit():     # -5 のような負の数も数として扱う
            stack.append(int(token))
        else:
            raise ValueError(f'知らないワード: {token}')


print(run('3 4 +'))    # [7]
print(run('10 3 -'))   # [7]
trace('3 4 + 2 *')

In [ ]:
# [CHECK] q1
need('WORDS', kind=dict)
need('evaluate', arity=2)
need('forth_eval', arity=2)
need('run', arity=1)

assert run('3 4 +') == [7]
assert run('10 3 -') == [7]        # 順序に注意
assert run('3 4 + 2 *') == [14]
assert run('2 3 4 * +') == [14]
assert run('20 4 /') == [5]
assert run('1 2 3') == [1, 2, 3]
assert run('-5 3 +') == [-2]
assert run('') == []

# 性質で確かめる
holds('数を並べたらそのまま積まれる',
      lambda a, b: run(f'{a} {b}') == [a, b], ints(2, lo=0, hi=99))
holds('足してから引くと元の数',
      lambda a, b: run(f'{a} {b} + {b} -') == [a], ints(2, lo=0, hi=99))
try:
    run('3 FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')

# WORDS に後から足したワードも、evaluate を変えずに使えること
def _neg(s):
    s.append(-s.pop())
WORDS['NEG'] = _neg
assert run('5 NEG') == [-5], '実行中にワードを増やせること'
del WORDS['NEG']
print('OK')

---
## 課題2: `.` とスタック操作ワード (15点)

**やること**: 下の表のワードの関数の中身を書く。表に入れる行 (`WORDS['DUP'] = dup` など) は書いてある。
**`evaluate` は変更しない**。表にワードを足すだけで、電卓が使えるワードが増える。

| ワード | 動作 | スタックの変化 |
|---|---|---|
| `.` | 一番上を**表示して**取り除く | (a → ) |
| `DUP` | 一番上を複製 | (a → a a) |
| `DROP` | 一番上を捨てる | (a → ) |
| `SWAP` | 上2つを入れ替え | (a b → b a) |

`(a b → b a)` は「左が実行前、右が実行後」のスタック。右端が一番上。

**入出力の例**:

```
run('5 DUP')        →  [5, 5]
run('1 2 DROP')     →  [1]
run('1 2 SWAP')     →  [2, 1]
run('42 .')         →  42 と表示され、スタックは []
```

**手順** (下のセルの `TODO` と同じ順): `dot`, `dup`, `drop`, `swap` の中身を1つずつ書く。

**ヒント**:

- 一番上は `stack[-1]` で、取り出さずに覗ける
- `DUP` は「一番上を覗いて、もう1つ積む」。`.` は「取り出して `print`」
- `SWAP` は、2つ取り出してから、逆の順に積み直す

**つまずきやすい点**:

- `stack.pop()` は値を**取り出す** (スタックから消える)。`stack[-1]` は**覗くだけ**。`DUP` で `pop` すると数が減る
- 「ドットは取り除くこと」「ドットは出力すること」と出たら、`.` の `pop` か `print` を忘れている

### 例題

次のセルを実行して、動きを見る。

ワードを1つ作って、`WORDS` に入れる形。

In [ ]:
# 例題
def neg(stack):
    stack.append(-stack.pop())


WORDS['NEG'] = neg
print(run('5 NEG'))    # [-5]
del WORDS['NEG']

In [ ]:
def dot(stack):
    print(stack.pop())


def dup(stack):
    stack.append(stack[-1])


def drop(stack):
    stack.pop()


def swap(stack):
    b, a = stack.pop(), stack.pop()
    stack.append(b)
    stack.append(a)


WORDS['.'] = dot
WORDS['DUP'] = dup
WORDS['DROP'] = drop
WORDS['SWAP'] = swap

print(run('1 2 SWAP'))    # [2, 1]
trace('5 DUP *')

In [ ]:
# [CHECK] q2
need('WORDS', kind=dict)
need('forth_eval', arity=2)

import io, contextlib

assert run('1 2 SWAP') == [2, 1]
assert run('1 2 swap') == [2, 1]     # 小文字でも動くこと
assert run('5 DUP') == [5, 5]
assert run('1 2 DROP') == [1]
assert run('5 dup *') == [25]
assert run('1 2 3 SWAP') == [1, 3, 2]
assert run('3 4 + 2 *') == [14]      # 課題1 が壊れていないこと
assert run('10 3 -') == [7]

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    s = run('42 .')
assert s == [], 'ドットは取り除くこと'
assert '42' in buf.getvalue(), 'ドットは出力すること'

try:
    run('FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')
print('OK')

---
## 課題3: ワードの定義を読む (15点)

ここから、電卓を**言語**にする。Forth では、次のように書くと新しいワードを作れる。

```
: SQUARE DUP * ;
```

`:` が「定義の始まり」、その次が名前 `SQUARE`、`;` までが本体 `DUP *`。
「SQUARE は、`DUP` して `*` することだ」と覚えておく。

**やること**: 定義を読み取る `read_definition(tokens, i)` の空いている所を埋める。
読み取るだけで、実行はしない (登録は用意済みの `forth_eval` がする)。

- `tokens`: 入力を空白で区切ったリスト
- `i`: `:` のある位置 (`tokens[i]` が `':'`)
- 返すもの: `(名前を大文字にしたもの, 本体のワードのリスト, ';' の次の位置)` の3つ組

**入出力の例**:

```
tokens = [':', 'SQUARE', 'DUP', '*', ';', '5', 'SQUARE']
位置      0      1        2     3    4    5     6

read_definition(tokens, 0)  →  ('SQUARE', ['DUP', '*'], 5)
```

**手順** (下のセルの `TODO` と同じ順。`;` の位置 `j` を探す行は書いてある):

1. `name`: `:` の次 `tokens[i + 1]` を、`.upper()` で大文字にする
2. `body`: 名前と `;` の間。スライス (リストの一部を `[始め:終わり]` で切り出すこと) で `tokens[i + 2:j]`

**ヒント**: 下の例題と同じことを、`i` を使って書けばよい。

**つまずきやすい点**:

- 名前を大文字にし忘れる。確認セルに「名前は大文字にすること」と出る
- `tokens[1]` や `tokens[2:j]` と、`i` を使わずに書くと、定義が先頭に無いときに失敗する (「2つ目の定義」で止まる)

### 例題

次のセルを実行して、動きを見る。

`;` の位置を探して、スライスで取り出す。

In [ ]:
# 例題
tokens = ': SQUARE DUP * ; 5 SQUARE'.split()
j = tokens.index(';', 0)
print(j)               # 4
print(tokens[1])       # SQUARE
print(tokens[2:j])     # ['DUP', '*']
print(j + 1)           # 5

In [ ]:
def read_definition(tokens, i):
    j = tokens.index(';', i)    # i より後ろで最初の ; の位置。無ければ ValueError
    name = tokens[i + 1].upper()
    body = tokens[i + 2:j]
    return name, body, j + 1


print(read_definition(': SQUARE DUP * ; 5 SQUARE'.split(), 0))   # ('SQUARE', ['DUP', '*'], 5)

In [ ]:
# [CHECK] q3
need('read_definition', arity=2)

assert read_definition(': SQUARE DUP * ; 5'.split(), 0) == ('SQUARE', ['DUP', '*'], 5)
assert read_definition('1 : inc 1 + ; inc'.split(), 1) == ('INC', ['1', '+'], 6), '名前は大文字にすること'
assert read_definition(': NOP ;'.split(), 0) == ('NOP', [], 3), '本体が空でもよい'
assert read_definition(': A B ; : C D ;'.split(), 4) == ('C', ['D'], 8), '2つ目の定義'

for bad in (': FOO DUP', ':'):
    try:
        read_definition(bad.split(), 0)
    except ValueError:
        pass
    else:
        raise AssertionError(f'{bad!r} は ValueError にすること')
print('OK')

---
## 課題4: 自分で定義したワードを実行する (15点)

用意済みの `forth_eval` は、`:` を見つけると `read_definition` で読み取り、
`USER_WORDS[名前] = 本体` に入れる。`USER_WORDS` の中身は、例えば `{'SQUARE': ['DUP', '*']}` になる。

**やること**: `SQUARE` のような自分で定義したワードが呼ばれたときに、
その本体を実行する `eval_user(stack, word)` を書く。

**入出力の例**:

```
run(': SQUARE DUP * ;  5 SQUARE')                                →  [25]
run(': DOUBLE DUP + ; : QUADRUPLE DOUBLE DOUBLE ;  5 QUADRUPLE')   →  [20]
```

**手順** (下のセルの `TODO`): 本体 `USER_WORDS[word]` を `forth_eval(stack, 本体)` に渡す。**1行で書ける**。

**ヒント**: `forth_eval` は文字列だけでなく、**ワードのリストも受け取れる** (下の例題)。
`QUADRUPLE` の本体の中の `DOUBLE` も、`forth_eval` がまた `eval_user` を呼ぶので、そのまま動く。

**つまずきやすい点**:

- 本体を `WORDS` に入れてしまう。確認セルに「ユーザ定義を WORDS に混ぜないこと」と出る。`eval_user` は表に入れず、実行するだけ
- `evaluate(stack, ...)` を呼ぶと、`DOUBLE` のような自分で定義したワードを知らないので止まる。`forth_eval` を呼ぶ

### 例題

次のセルを実行して、動きを見る。

リストを渡しても `forth_eval` は動く。

In [ ]:
# 例題
s = [5]
forth_eval(s, ['DUP', '*'])
print(s)    # [25]

In [ ]:
def eval_user(stack, word):
    forth_eval(stack, USER_WORDS[word])


USER_WORDS.clear()
print(run(': SQUARE DUP * ;  5 SQUARE'))                                  # [25]
print(run(': DOUBLE DUP + ; : QUADRUPLE DOUBLE DOUBLE ;  5 QUADRUPLE'))   # [20]

In [ ]:
# [CHECK] q4
need('USER_WORDS', kind=dict)
need('eval_user', arity=2)
need('forth_eval', arity=2)

USER_WORDS.clear(); assert run(': SQUARE DUP * ; 5 SQUARE') == [25]
USER_WORDS.clear(); assert run(': DOUBLE DUP + ; : QUAD DOUBLE DOUBLE ; 5 QUAD') == [20]
USER_WORDS.clear(); assert run(': INC 1 + ; 10 INC INC INC') == [13]
USER_WORDS.clear(); assert run('1 : ADD2 2 + ; ADD2 ADD2') == [5]
USER_WORDS.clear(); assert run(': SQ DUP * ; 3 SQ SQ') == [81]

# 定義しただけでは何もスタックに積まれない
USER_WORDS.clear(); assert run(': NOP ;') == []

# 大文字小文字を区別しないこと
USER_WORDS.clear(); assert run(': SQUARE DUP * ; 4 square') == [16]

# 組み込みが壊れていないこと
USER_WORDS.clear()
assert run('3 4 + 2 *') == [14]
assert run('1 2 SWAP') == [2, 1]
# ユーザ定義は USER_WORDS に入れ、組み込みの表は汚さないこと
USER_WORDS.clear()
_before = set(WORDS)
run(': SQUARE DUP * ;')
assert set(WORDS) == _before, 'ユーザ定義を WORDS に混ぜないこと'
assert len(USER_WORDS) == 1, f'USER_WORDS が {len(USER_WORDS)} 件'
note('ユーザ定義ワードは組み込みと別の表に入っている')

print('OK')

---
## 課題5: 対話ループ (REPL) (15点)

**REPL** (Read–Eval–Print Loop) は、1行読んで、実行して、結果を表示する、を繰り返す仕組み (Python の `>>>` と同じ)。
採点できるように、`input()` で読むかわりに、**入力行のリストを引数で受け取る**形で書く。

**やること**: `repl(lines)` の空いている所を埋める。`lines` は入力行のリスト。
返すのは、**各行に対する出力の文字列のリスト**。

| 入力行 | 返すもの |
|---|---|
| 普通の行 | 実行後のスタックを `str()` で文字列にしたもの (例: `'[7]'`) |
| `bye` | そこで終了。それ以降の行は処理しない |
| エラーが起きた行 | `'error: ' + エラーメッセージ`。**ループは続ける** |

**入出力の例**:

```
repl(['3 4 +', '2 *'])          →  ['[7]', '[14]']   (2行目は 1行目のスタック [7] の続き)
repl(['1 2 +', 'bye', '999'])   →  ['[3]']           (bye の後は無視)
repl(['1 2 +', 'FOO', '10 +'])  →  ['[3]', 'error: ...', '[13]']
```

**手順** (下のセルの `TODO` と同じ順):

1. `if` の条件: 行が `bye` かどうか。前後の空白や大文字は `line.strip().lower() == 'bye'` でそろえる
2. `try:` の中: `str(stack)` を `outputs` に足す (`outputs.append(...)`)
3. `except` の中: `'error: ' + str(e)` を `outputs` に足す

**ヒント**: 下の例題が、ほぼそのままの形。

**つまずきやすい点**:

- `stack = []` をループの**中**に移すと、行ごとにスタックが空に戻る。「エラーのあともスタックが保たれること」と出る。`stack` はループの外 (書いてある所) のまま
- `bye` を `try` の中で判定すると、`bye` 自体が知らないワードとしてエラーになる。`try` の前 (書いてある所) で判定する

### 例題

次のセルを実行して、動きを見る。

1行ずつ、失敗しても続ける形。

In [ ]:
# 例題
outputs = []
stack = []
for line in ['3 4 +', 'FOO']:
    try:
        forth_eval(stack, line)
        outputs.append(str(stack))
    except Exception as e:
        outputs.append('error: ' + str(e))
print(outputs)    # ['[7]', 'error: ...']

`stack` をループの外で作っているので、値は行をまたいで残る。

In [ ]:
def repl(lines):
    stack = []
    outputs = []
    for line in lines:
        if line.strip().lower() == 'bye':
            break
        try:
            forth_eval(stack, line)
            outputs.append(str(stack))
        except Exception as e:
            outputs.append('error: ' + str(e))
    return outputs


USER_WORDS.clear()
for out in repl(['3 4 +', '2 *', 'FOO', ': SQ DUP * ;', 'SQ', 'bye', '999']):
    print(out)

In [ ]:
# 対話用 (採点しない)。実行すると入力待ちになり、bye で終わる
def repl_interactive():
    stack = []
    while True:
        line = input('> ')
        if line.strip().lower() == 'bye':
            break
        try:
            forth_eval(stack, line)
            print(stack)
        except Exception as e:
            print('error:', e)


# repl_interactive()   ← コメントを外すと対話できる

In [ ]:
# [CHECK] q5
need('repl', arity=1)

USER_WORDS.clear()
out = repl(['3 4 +', '2 *'])
assert out == ['[7]', '[14]'], out

# bye で止まり、それ以降は処理しない
USER_WORDS.clear()
out = repl(['1 2 +', 'bye', '999'])
assert out == ['[3]'], out

# エラーでもループは続く
USER_WORDS.clear()
out = repl(['1 2 +', 'FOO', '10 +'])
assert len(out) == 3, out
assert out[0] == '[3]'
assert out[1].startswith('error:'), out[1]
assert out[2] == '[13]', 'エラーのあともスタックが保たれること'

# ユーザ定義ワードも行をまたいで残る
USER_WORDS.clear()
out = repl([': SQ DUP * ;', '5 SQ'])
assert out == ['[]', '[25]'], out

assert repl([]) == []
assert repl(['bye']) == []
print('OK')

---
# 発展課題

時間に余裕があれば取り組む。基本課題 (課題1〜5) だけで 80 点。

## 発展1: 比較演算子 (10点)

**やること**: 比較のワード `<` `>` `=` を `WORDS` に入れる。`<` は例題に書いてあるので、`>` と `=` を書く。

Forth では、**真 (正しい) を `-1`、偽 (正しくない) を `0`** で表す。

**入出力の例**:

```
run('3 4 <')    →  [-1]    (3 < 4 は正しい)
run('3 4 >')    →  [0]
run('4 4 =')    →  [-1]
```

**手順**: `less` をまねて `greater` と `equal` を書き、`WORDS['>']`, `WORDS['=']` に入れる。

**つまずきやすい点**: Python の `True` をそのまま積むと `1` になる。`-1 if a > b else 0` のように `-1` / `0` に直す。

### 例題

次のセルを実行して、動きを見る。

比較の結果を Forth の真偽値 (`-1` / `0`) に直す。

In [ ]:
# 例題
def less(stack):
    b, a = stack.pop(), stack.pop()
    stack.append(-1 if a < b else 0)


WORDS['<'] = less

`>` と `=` も同じ形。

In [ ]:
def greater(stack):
    b, a = stack.pop(), stack.pop()
    stack.append(-1 if a > b else 0)


def equal(stack):
    b, a = stack.pop(), stack.pop()
    stack.append(-1 if a == b else 0)


WORDS['>'] = greater
WORDS['='] = equal


In [ ]:
# [CHECK] b1
USER_WORDS.clear()
assert run('3 4 <') == [-1]
assert run('4 3 <') == [0]
assert run('3 4 >') == [0]
assert run('4 3 >') == [-1]
assert run('4 4 =') == [-1]
assert run('3 4 =') == [0]
assert run('4 4 <') == [0] and run('4 4 >') == [0]
assert run('1 2 + 3 =') == [-1]
print('OK')

## 発展2: 条件分岐 (10点)

**やること**: `IF ... ELSE ... THEN` を使えるようにする。
`IF` を読んだときにスタックの一番上にある値が 0 以外なら `IF` と `ELSE` の間を、
0 なら `ELSE` と `THEN` の間を実行する (`ELSE` は省略できる)。

**入出力の例**:

```
run('1 IF 10 ELSE 20 THEN')    →  [10]
run('0 IF 10 ELSE 20 THEN')    →  [20]
: ABS DUP 0 < IF 0 SWAP - THEN ;
-5 ABS .                       →  5
```

`IF` から `THEN` までを切り分ける `split_if` は、例題に書いてある。
下のセルの `forth_eval` (課題0 と同じもの) の `TODO` の所に、`IF` の場合を足す。

**手順**:

1. `key == 'IF'` なら、`yes, no, i = split_if(tokens, i)` で、真のときの部分・偽のときの部分・`THEN` の次の位置を受け取る
2. 条件 `stack.pop()` が 0 以外なら `yes` を、0 なら `no` を `forth_eval(stack, ...)` に渡す
3. `continue` で、次のワードへ進む (`:` の場合と同じ形)

**つまずきやすい点**:

- 条件を `pop` せずに `stack[-1]` で見ると、条件の値がスタックに残る。`run('1 IF 42 THEN')` が `[1, 42]` になる
- `continue` を忘れると、`i += 1` で `THEN` の次のワードを飛ばしてしまう

### 例題

次のセルを実行して、動きを見る。

`IF` の中にまた `IF` があっても (入れ子)、深さを数えて、対応する `ELSE` と `THEN` を探す。

In [ ]:
# 例題
def split_if(tokens, i):
    """tokens[i] == 'IF' のとき、(真のときの部分, 偽のときの部分, THEN の次の位置) を返す"""
    depth = 1
    else_at = None
    j = i + 1
    while j < len(tokens):
        word = tokens[j].upper()
        if word == 'IF':
            depth += 1
        elif word == 'ELSE' and depth == 1:
            else_at = j
        elif word == 'THEN':
            depth -= 1
            if depth == 0:
                break
        j += 1
    else:
        raise ValueError('THEN がない')
    if else_at is None:
        return tokens[i + 1:j], [], j + 1
    return tokens[i + 1:else_at], tokens[else_at + 1:j], j + 1


print(split_if('1 IF 10 ELSE 20 THEN 5'.split(), 1))   # (['10'], ['20'], 6)

In [ ]:
def forth_eval(stack, text):
    """ワードを左から1つずつ実行する。text は文字列でもワードのリストでもよい"""
    tokens = text.split() if isinstance(text, str) else text
    i = 0  # 今見ているワードの位置
    while i < len(tokens):
        token = tokens[i]
        key = token.upper()
        # ':' なら定義を読み取り、USER_WORDS に登録する
        if token == ':':
            name, body, i = read_definition(tokens, i)
            USER_WORDS[name] = body
            continue
        if key == 'IF':
            yes, no, i = split_if(tokens, i)
            forth_eval(stack, yes if stack.pop() != 0 else no)
            continue
        # 自分で定義したワードなら本体を実行。それ以外は evaluate に任せる
        if key in USER_WORDS:
            eval_user(stack, key)
        else:
            evaluate(stack, key)
        i += 1

In [ ]:
# [CHECK] b2
USER_WORDS.clear()
assert run('1 IF 42 THEN') == [42]
assert run('0 IF 42 THEN') == []
assert run('1 IF 10 ELSE 20 THEN') == [10]
assert run('0 IF 10 ELSE 20 THEN') == [20]
assert run('-1 IF 7 THEN') == [7], '0 以外はすべて真'

USER_WORDS.clear()
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; -5 ABS') == [5]
USER_WORDS.clear()
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; 5 ABS') == [5]

# 入れ子
USER_WORDS.clear()
assert run('1 IF 1 IF 7 ELSE 8 THEN ELSE 9 THEN') == [7]
assert run('1 IF 0 IF 7 ELSE 8 THEN ELSE 9 THEN') == [8]
assert run('0 IF 1 IF 7 ELSE 8 THEN ELSE 9 THEN') == [9]

# 条件分岐の外側は普通に続く
assert run('1 IF 10 THEN 5 +') == [15]
print('OK')

---
## 自己確認

`score()` で全部の確認をまとめて実行する。何度やってもよい。
`NG` の行に、次に何をすればいいかが出る。

In [ ]:
# [SKIP] 採点の対象外
score()

---
## 提出

提出は kibaco から。`score()` は下書きの確認用で、提出にはならない。

1. 一番上から全部実行し、出力を残したまま保存する
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても、教員の採点は別の検査で行う。